<a href="https://colab.research.google.com/github/IdaCy/ParaphrX/blob/main/notebooks/semantics_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check Python version (optional):
import sys
print("Python version:", sys.version)

# Get installations
!pip install --quiet torch numpy matplotlib scikit-learn pandas
!pip install --quiet huggingface_hub transformers

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# If you want to check GPU usage:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Python version: 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.8 MB/s eta 0:00:00
Using device: cuda


In [2]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()
%ls

sample_data/


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# After running this cell, follow the link to grant Colab access to your Google Drive.

Mounted at /content/drive


In [5]:
!git clone https://github.com/IdaCy/polAItness_internals.git
%cd polAItness_internals

Cloning into 'polAItness_internals'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 334 (delta 22), reused 28 (delta 6), pack-reused 287 (from 1)
Receiving objects: 100% (334/334), 474.49 KiB | 2.50 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/polAItness_internals


In [6]:
!pip install huggingface_hub --quiet

from huggingface_hub import notebook_login

# This will prompt you in Colab to enter your HF token or log in directly
notebook_login()

In [10]:
import logging
from collab_modules.logging import init_logger
logging.basicConfig(level=logging.INFO)

logger = init_logger(
    log_file="logs/progress.log",
    console_level=logging.WARNING,  # only warnings to console
    file_level=logging.DEBUG        # full debug info in the file
)

In [11]:
from collab_modules.indexed_inference import (
    load_model,
    load_json_prompts,
    run_inf
)

In [9]:
model, tokenizer = load_model(
    model_name="google/gemma-2-9b-it",
    use_bfloat16=True,
    hf_token=None,
    logger=logger
)

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:862: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [10]:
!git pull

Already up to date.


In [11]:
# Inference

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key="instruction",
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/instruction_normal",
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


In [12]:
# Inference
prompt_key = "instruction_polite"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

In [13]:
# Inference
prompt_key = "instruction_urgent"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

In [14]:
# Inference
prompt_key = "instruction_rude"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

In [15]:
# Inference
prompt_key = "instruction_longer"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

In [16]:
# Inference
prompt_key = "instruction_longest"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

In [17]:
# Inference
prompt_key = "instruction_shorter"

data = load_json_prompts(
    file_path="prompts/instructions.json",
    prompt_key=prompt_key,
    max_samples=None,
    logger=logger
)

# 3. Rerun inference
run_inf(
    model=model,
    tokenizer=tokenizer,
    data=data,
    batch_size=16,
    output_dir="/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/" + prompt_key,
    extract_hidden_layers=[0, 5, 10, 15, 20, 25],
    extract_attention_layers=[0, 5, 10, 15, 20, 25],
    max_seq_length=2048,
    logger=logger,
    generation_kwargs={
        "do_sample": True,
        "max_new_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    }
)

## Reward Model Scoring

This uses OpenAssistant/reward-model-deberta-v3-large-v2 from   
https://huggingface.co/OpenAssistant/reward-model-deberta-v3-large-v2
to assess how good predictions are

In [19]:
!pip show accelerate

Name: accelerate
Version: 1.5.2
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /usr/local/lib/python3.11/dist-packages
Requires: huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: peft


In [20]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.68 KiB | 2.68 MiB/s, done.
From https://github.com/IdaCy/polAItness_internals
   d465428..80e4b8b  main       -> origin/main
Updating d465428..80e4b8b
Fast-forward
 collab_modules/reward_model.py | 221 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 221 insertions(+)
 create mode 100644 collab_modules/reward_model.py


In [8]:
from collab_modules.reward_model import (
    load_reward_model,
    score_predictions_and_save
)

In [14]:
pt_path = "/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/"
analy_path = "/content/drive/MyDrive/polAItness_internals/analyses/"

pt_dir = "/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/instruction_normal"
raw_csv_path = "/content/drive/MyDrive/polAItness_internals/analyses/scored_predictions.csv"
agg_base_path = "/content/drive/MyDrive/polAItness_internals/analyses/scored_aggregates"  # produce .json & .txt
rm_name = "OpenAssistant/reward-model-deberta-v3-large-v2"

# logger
#logger = logging.getLogger("ScoringLogger")
#logger.setLevel(logging.WARNING)

# reward model pipeline
rm_pipeline = load_reward_model(
    model_name=rm_name,
    device=0,
    logger=logger
)

Device set to use cuda:0


In [28]:
# loads all .pt, scores them, saves results
scored_list = score_predictions_and_save(
    pt_dir=pt_dir,
    rm_pipeline=rm_pipeline,
    raw_csv_path=raw_csv_path,
    agg_base_path=agg_base_path,
    logger=logger
)

# aggregated average
if len(scored_list) > 0:
    avg_score = sum(x['reward_score'] for x in scored_list) / len(scored_list)
    print(f"Average score over {len(scored_list)} predictions: {avg_score:.4f}")
else:
    print("No scored items found!")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Average score over 227 predictions: 0.2473


In [13]:
pt_dir = "/content/drive/MyDrive/polAItness_internals/output/instr_extractions/gemma-2-9b-it/instruction_polite"

# loads all .pt, scores them, saves results
scored_list = score_predictions_and_save(
    pt_dir=pt_dir,
    rm_pipeline=rm_pipeline,
    raw_csv_path=raw_csv_path,
    agg_base_path=agg_base_path,
    logger=logger
)

# aggregated average
if len(scored_list) > 0:
    avg_score = sum(x['reward_score'] for x in scored_list) / len(scored_list)
    print(f"Average score over {len(scored_list)} predictions: {avg_score:.4f}")
else:
    print("No scored items found!")

[WARNING] No scores found, skipping aggregates.


No scored items found!
